# RAG Based AI AGENT

In [2]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("sql_notes.pdf")
user_pdf = loader.load()

print(user_pdf)

[Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '', 'source': 'sql_notes.pdf', 'total_pages': 27, 'page': 0, 'page_label': '1'}, page_content="DBMS & SQL NOTES\nDatabase:A database is a collection of related datawhich represents some aspect of the realworld. A database system is designed to be built and populated with data for a certain task.\nDatabase Management System (DBMS)is a software forstoring and retrieving users' data whileconsidering appropriate security measures. It consists of a group of programs which manipulatethe database. The DBMS accepts the request for data from an application and instructs theoperating system to provide the specific data. In large systems, a DBMS helps users and otherthird-party software to store and retrieve data.\nDatabase management systems were developed to handle the following difficulties oftypical File-processing systems supported by conventional operating systems.1. Data redundancy and inconsistency2. Difficulty i

In [3]:
for i in user_pdf:
    print(i.model_json_schema())

{'description': 'Class for storing a piece of text and associated metadata.\n\n!!! note\n\n    `Document` is for **retrieval workflows**, not chat I/O. For sending text\n    to an LLM in a conversation, use message types from `langchain.messages`.\n\nExample:\n    ```python\n    from langchain_core.documents import Document\n\n    document = Document(\n        page_content="Hello, world!", metadata={"source": "https://example.com"}\n    )\n    ```', 'properties': {'id': {'anyOf': [{'type': 'string'}, {'type': 'null'}], 'default': None, 'title': 'Id'}, 'metadata': {'additionalProperties': True, 'title': 'Metadata', 'type': 'object'}, 'page_content': {'title': 'Page Content', 'type': 'string'}, 'type': {'const': 'Document', 'default': 'Document', 'title': 'Type', 'type': 'string'}}, 'required': ['page_content'], 'title': 'Document', 'type': 'object'}
{'description': 'Class for storing a piece of text and associated metadata.\n\n!!! note\n\n    `Document` is for **retrieval workflows**, n

## Split into Chunks

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
     chunk_size=1000,
    chunk_overlap=200
)

user_docs = splitter.split_documents(user_pdf)

print(len(user_docs))

48


In [5]:
from huggingface_hub import login
import os

hugging_face_api_key = os.getenv("HUGGINGFACE_API_KEY")

login(hugging_face_api_key)

## Create Embeddings

In [6]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1690.23it/s]


## Store in Vector DB

In [7]:
from langchain_community.vectorstores import Chroma

vectorStore = Chroma.from_documents(
    documents=user_docs,
    embedding=embeddings,
    persist_directory="chroma_db"
)

In [8]:
vectorStore

## Create Retriever

In [9]:
retriever = vectorStore.as_retriever(
    search_kwargs={"k": 3}
)

## Initialize LLM

In [10]:
from langchain_groq import ChatGroq
from dotenv import  load_dotenv

load_dotenv()

llm_model = ChatGroq(
    model="openai/gpt-oss-20b"
)

## Create RAG State

In [11]:
from typing_extensions import TypedDict

class RAGState(TypedDict):
    question:str
    context:str
    answer:str


## Create a Node Definition of RAG

In [12]:
def rag_retriever_node(state: RAGState):

    question = state["question"]

    retrieved_docs = retriever.invoke(question)

    context = "\n\n".join(
        doc.page_content
        for doc in retrieved_docs
    )
    return {
        "context": context
    }

## Answer Generator

In [13]:
def answer_node(state: RAGState):

    question = state["question"]

    context = state["context"]

    response = llm_model.invoke(
        f"""
        Answer the question using ONLY
        the provided context.

        Context:
        {context}

        Question:
        {question}
        """
    )

    return {
        "answer": response.content
    }

In [14]:
from langgraph.graph import StateGraph,START,END

builder = StateGraph(RAGState)

## Add Nodes

In [15]:
builder.add_node(
    "retrieve",
    rag_retriever_node
)

builder.add_node(
    "answer",
    answer_node
)

## Add Edges

In [16]:
builder.add_edge(
    START,
    "retrieve"
)

builder.add_edge(
    "retrieve",
    "answer"
)

builder.add_edge(
    "answer",
    END
)

## Compile Graph

In [17]:
graph = builder.compile()

## Ask Questions

In [18]:
# while True:
#     user_input = input("Enter a question: ")
#     if user_input == "quit":
#         break

In [23]:
result = graph.invoke({
    "question":
    "types of joins in with example and for each concept should be an exaple"
})

In [24]:
print(result["answer"])

**Types of JOINs (from the supplied context)**  

| JOIN type | Purpose | Example (as shown in the context) |
|-----------|---------|----------------------------------|
| **INNER JOIN** | Returns only rows that have matching values in both tables. | `SELECT Orders.OrderID, Customers.CustomerName FROM Orders INNER JOIN Customers ON Orders.CustomerID = Customers.CustomerID;` |
| **LEFT (OUTER) JOIN** | Returns all rows from the left table and the matched rows from the right table; unmatched rows from the right table are `NULL`. *(No example is provided in the context.)* |  |
| **RIGHT JOIN** | Returns all rows from the right table and the matched rows from the left table; unmatched rows from the left table are `NULL`. | `SELECT Orders.OrderID, Employees.LastName, Employees.FirstName FROM Orders RIGHT JOIN Employees ON Orders.EmployeeID = Employees.EmployeeID ORDER BY Orders.OrderID;` |
| **FULL (OUTER) JOIN** | Returns all rows when there is a match in either the left or right table; unm

In [28]:
tools = ["add", "multiply", "divide"]
tools_by_name = {tool: tool for tool in tools}
print(tools_by_name)

{'add': 'add', 'multiply': 'multiply', 'divide': 'divide'}
